In [1]:
import torch, sys, os, numpy as np
from PIL import Image

RDT2_DIR     = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"
CHECKPOINT   = "/home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-1000"
NORMALIZER   = "/home/rishabh/Downloads/umi-pipeline-training/normalizer.pt"
DATASET_PATH = "/home/rishabh/Downloads/umi-pipeline-training/replay_buffer.zarr"
INSTRUCTION  = "pick up the marker and place it in the box"

# Add ALL paths
sys.path.insert(0, RDT2_DIR)
sys.path.insert(0, os.path.join(RDT2_DIR, 'vqvae'))

os.environ['TRANSFORMERS_ATTN_IMPLEMENTATION'] = 'eager'
os.environ['PYTORCH_CUDA_ALLOC_CONF']          = 'expandable_segments:True'
os.environ['CUDA_LAUNCH_BLOCKING']             = '1'

print("✅ Paths set")
print(f"   RDT2:       {RDT2_DIR}")
print(f"   Checkpoint: {CHECKPOINT}")

✅ Paths set
   RDT2:       /home/rishabh/Downloads/umi-pipeline-training/RDT2
   Checkpoint: /home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-1000


In [2]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import PeftModel
from models.normalizer.normalizer import LinearNormalizer
from models.multivqvae import MultiVQVAE   # correct path!

print("[1/4] Loading base model...")
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "robotics-diffusion-transformer/RDT2-VQ",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
    device_map="cuda"
)

print("[2/4] Loading your fine-tuned adapter...")
model = PeftModel.from_pretrained(base, CHECKPOINT)
model.eval()

print("[3/4] Loading processor...")
processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct", use_fast=True
)

print("[4/4] Loading VAE + normalizer...")
vae = MultiVQVAE.from_pretrained(
    "robotics-diffusion-transformer/RVQActionTokenizer"
).cuda().eval()

normalizer = LinearNormalizer.load(NORMALIZER)

print("\n✅ ALL MODELS LOADED!")

/home/rishabh/Downloads/umi-pipeline-training/umi_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[1/4] Loading base model...


Loading checkpoint shards: 100%|██████████| 4/4 [00:01<00:00,  2.45it/s]


[2/4] Loading your fine-tuned adapter...
[3/4] Loading processor...
[4/4] Loading VAE + normalizer...
LinearNormalizer loaded from /home/rishabh/Downloads/umi-pipeline-training/normalizer.pt

✅ ALL MODELS LOADED!


In [3]:
import zarr

print("Loading image from dataset...")
root    = zarr.open(DATASET_PATH, mode='r')
img_arr = np.array(root['data']['camera0_rgb'][500], dtype=np.uint8)
img     = Image.fromarray(img_arr).resize((768, 384))
img.save("/tmp/test_frame.jpg")

print(f"✅ Image: {img_arr.shape} → 768x384")
print(f"   Saved: /tmp/test_frame.jpg")

Loading image from dataset...
✅ Image: (224, 224, 3) → 768x384
   Saved: /tmp/test_frame.jpg


In [4]:
print(f"Running inference: '{INSTRUCTION}'")

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": img},
        {"type": "text",  "text": INSTRUCTION}
    ]
}]

text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = processor(
    text=[text], images=[img], return_tensors="pt"
).to("cuda", torch.bfloat16)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
    )

input_len = inputs["input_ids"].shape[1]
generated = output_ids[:, input_len:]

print(f"✅ Generated {generated.shape[1]} action tokens")
print(f"   Token IDs: {generated[0].tolist()}")

Running inference: 'pick up the marker and place it in the box'


/home/rishabh/Downloads/umi-pipeline-training/umi_env/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


✅ Generated 30 action tokens
   Token IDs: [151650, 151534, 150700, 150676, 150677, 150675, 151604, 150623, 150837, 150840, 151402, 150807, 151285, 150973, 151612, 150634, 150759, 150869, 150656, 151086, 151101, 151378, 151160, 150693, 150664, 150858, 151627, 151232, 151651, 151645]


In [ ]:
fake_ids = torch.tensor([[7,8,2,4,3,5,1,0,6,9,10,11,12,13,14,15,16,17]]).cuda()

with torch.no_grad():
    out = vae.decode(fake_ids)

print(out.shape)


RuntimeError: Calculated padded input size per channel: (2). Kernel size: (3). Kernel size can't be greater than actual input size

: 

In [5]:
print("Decoding action tokens → real action values...")

with torch.no_grad():

    # ✅ Move to CPU FIRST (prevents CUDA assert)
    token_ids_cpu = generated[0].cpu()

    VOCAB_OFFSET = 151643

    # keep only valid action tokens
    action_ids_cpu = token_ids_cpu[token_ids_cpu >= VOCAB_OFFSET] - VOCAB_OFFSET

    print(f"Valid action ids count: {action_ids_cpu.numel()}")
    print(f"Filtered IDs: {action_ids_cpu.tolist()}")

    try:
        # send ONLY clean ids to GPU
        action_ids = action_ids_cpu.unsqueeze(0).cuda()

        action_embedding = vae.decode(action_ids)

        action_raw = action_embedding[0,0].cpu().float().numpy()
        print("Raw action:", np.round(action_raw,4))

        action_t = torch.from_numpy(action_raw).unsqueeze(0)
        action = normalizer['action'].unnormalize(action_t).squeeze(0).numpy()

        print("✅ VAE decode SUCCESS")

    except Exception as e:
        print("⚠️ VAE decode failed:", e)
        print("Using fallback decode...")

        token_ids_np = generated[0].cpu().numpy()
        action_idx   = token_ids_np - VOCAB_OFFSET
        NUM_CODES    = 512
        action_norm  = action_idx / NUM_CODES

        action       = np.zeros(7)
        n            = min(len(action_norm),7)
        action[:n]   = action_norm[:n]
        action[6]    = np.clip(action[6],0,1)


Decoding action tokens → real action values...
Valid action ids count: 3
Filtered IDs: [7, 8, 2]
⚠️ VAE decode failed: shape '[1, -1, 6]' is invalid for input of size 3
Using fallback decode...
